In [2]:
# fastfm 라이브러리를 설치
!pip install fastfm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 6.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for fastfm: filename=fastFM-0.2.10-cp312-cp312-linux_x86_64.whl size=660958 sha256=6c6e7a73a23a64e8e8c3931a13b40fc8be81287c276a6c46e966161f9267ff65
  Stored in directory: /root/.cache/pip/wheels/05/3e/62/4910464465a0df01d7eb87bf5101d306ee89b2d690633b70f4
Successfully built fastfm


In [4]:
# -nc 옵션은 파일이 이미 존재하면 다시 다운로드하지 않도록
!wget https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_comics_graphic.json.gz

--2026-05-31 05:54:27--  https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_comics_graphic.json.gz
Resolving mcauleylab.ucsd.edu (mcauleylab.ucsd.edu)... 137.110.161.5
Connecting to mcauleylab.ucsd.edu (mcauleylab.ucsd.edu)|137.110.161.5|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 146702530 (140M) [application/gzip]
Saving to: ‘goodreads_reviews_comics_graphic.json.gz’

goodreads_reviews_c 100%[===================>] 139.91M  2.94MB/s    in 46s     

2026-05-31 05:55:15 (3.01 MB/s) - ‘goodreads_reviews_comics_graphic.json.gz’ saved [146702530/146702530]



In [10]:

import gzip
import json

# ID를 정수 인덱스로 매핑하기 위한 딕셔너리
user_ids = {}
book_ids = {}

# 파싱된 데이터를 저장할 리스트
data = []

# 'rt'는 텍스트 모드
with gzip.open('goodreads_reviews_comics_graphic.json.gz',"rt") as f:
  for line in f:
    # 각 줄을 JSON 객체로 파싱
    record = json.loads(line)

    # 레코드에서 필요한 정보(사용자 ID, 책 ID, 평점, 투표수, 댓글수)를 추출
    uid = record['user_id']
    bid = record['book_id']
    rating = record['rating']
    n_votes = record['n_votes']
    n_comments = record['n_comments']

    # ID가 딕셔너리에 없으면 새로 추가하고 인덱스를 할당
    if uid not in user_ids:
      user_ids[uid] = len(user_ids)
    if bid not in book_ids:
      book_ids[bid] = len(book_ids)

    # 처리된 데이터를 'data' 리스트에 추가
    # 여기에는 매핑된 사용자 ID, 책 ID, 투표수, 댓글수, 평점이 포함
    data.append((user_ids[uid], book_ids[bid], n_votes, n_comments, rating))

n_users = len(user_ids)
n_books = len(book_ids)

print(n_users, n_books)


59347 89311
(0, 0, 0, 0, 3)


In [ ]:
from scipy.sparse import csr_matrix
import numpy as np
import random

random.shuffle(data)

rows = []
cols = []
values = []

# 투표수와 댓글수를 위한 특성(feature) 인덱스를 설정
# 사용자 및 책 ID 인덱스 다음에 위치
n_votes_idx = n_users+n_books
n_comments_idx = n_users+n_books+1

# 'data' 리스트의 각 항목을 반복하여 희소 행렬의 구성 요소를 채움
for rowid, (uid, bid, n_votes, n_comments, rating) in enumerate(data):
  # 각 레코드에 대해 사용자 ID, 책 ID, 투표수, 댓글수를 특성으로 추가
  rows.extend([rowid, rowid, rowid, rowid])
  cols.extend([uid, n_users+bid, n_votes_idx, n_comments_idx])
  values.extend([1,1,n_votes,n_comments])

# CSR 형식의 희소 행렬 'x'를 생성
x = csr_matrix((values,(rows,cols)))
# 평점(rating)을 numpy 배열 'y'로 생성. 모델의 타겟 변수
y = np.array([rating for _,_,_,_, rating in data], dtype=np.float64)

x_train, y_train = x[:400000], y[:400000]
x_test, y_test = x[400000:], y[400000:]

In [ ]:
# fastFM 라이브러리에서 ALS(Alternating Least Squares) 방식의 Factorization Machine 회귀 모델을 임포트
from fastFM import als

fm = als.FMRegression(n_iter=1000, init_stdev=0.1, rank=5, l2_reg_w=0.1, l2_reg_V=0.5)
fm.fit(x_train, y_train)

FMRegression(l2_reg_V=0.5, n_iter=1000, rank=5)

In [ ]:
# 학습된 FM 모델을 사용하여 테스트 세트(x_test)에 대한 평점 예측을 수행
y_pred = fm.predict(x_test)
print(y_pred[:10])
print(y_test[:10])
# 예측 성능을 평가하기 위해 RMSE(Root Mean Squared Error)를 계산
print("RMSE: ", ((y_test - y_pred)**2).mean() ** 0.5)

[3.16159944 3.97931786 5.24564895 3.01652874 4.17435626 3.94326701
 4.52619545 4.10344158 3.52648363 4.10802057]
[4. 4. 5. 3. 5. 4. 3. 4. 5. 5.]
RMSE:  4.916403390982525
